
# Experiment 03 — FedAvg under Dirichlet Label Skew (MNIST)

This notebook studies **FedAvg under tunable statistical heterogeneity** generated with a
class-wise Dirichlet partition.

The central question is:

> **How does FedAvg change as client label distributions become progressively more heterogeneous?**

For each class $c$, its training examples are distributed across $K$ clients according to

$$
(p_{c1}, \ldots, p_{cK})
\sim \operatorname{Dirichlet}(\alpha,\ldots,\alpha).
$$

The concentration parameter $\alpha$ controls heterogeneity:

- larger $\alpha$ → more even, more IID-like class allocation;
- smaller $\alpha$ → stronger class concentration on a small number of clients.

### Experimental design

We initially sweep

$$
\alpha \in \{10,\;1,\;0.5,\;0.1\}
$$

while holding the local training budget fixed at

$$
E=1,\qquad T=20.
$$

This isolates the effect of **heterogeneity severity** instead of simultaneously changing
both $\alpha$ and $E$.

### Controlled variables

Across the IID, shard, and Dirichlet experiments we keep fixed:

- MNIST preprocessing;
- 5 clients;
- `SimpleMLP(784 → 128 → 10)`;
- SGD with learning rate $0.01$;
- batch size $64$;
- the same test set;
- the same saved initial checkpoint $w_0$;
- full client participation ($C=1$).

> **Important interpretation note.** Standard class-wise Dirichlet partitioning primarily
> controls label heterogeneity, but it can also create unequal client dataset sizes. We record
> that quantity imbalance explicitly instead of pretending the experiment isolates label skew
> perfectly.


## 1. Setup

In [ ]:

from pathlib import Path
import os
import sys


def find_repo_root():
    """Find the repository root from VS Code, Kaggle, or Colab."""
    cwd = Path.cwd().resolve()

    candidates = [cwd, *cwd.parents]
    candidates += [
        Path("/kaggle/working/federated-learning-under-heterogeneity"),
        Path("/content/federated-learning-under-heterogeneity"),
    ]

    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve()

    raise RuntimeError(
        "Repository root not found.\n"
        "Open the 'federated-learning-under-heterogeneity' folder in VS Code, "
        "or clone/cd into the repository before running this notebook."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root  :", REPO_ROOT)
print("Working dir:", Path.cwd())
print("Python     :", sys.executable)


In [ ]:

import copy
import json
import random
import time
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.models import SimpleMLP
from src.training import evaluate, federated_train

# Import through the module first so a missing local implementation gives a useful error.
from src import data as data_module

if not hasattr(data_module, "partition_dirichlet"):
    raise ImportError(
        "partition_dirichlet() is missing from src/data.py.\n"
        "Save your local Dirichlet implementation there before running this notebook."
    )

partition_dirichlet = data_module.partition_dirichlet
create_client_loaders = data_module.create_client_loaders
client_label_counts = data_module.client_label_counts

print("src.data loaded from:", data_module.__file__)
print("Imports successful.")



### Implementation map

This notebook contains **experiment logic only**.

Reusable code stays under `src/`:

- `src/models.py` — model definition;
- `src/data.py` — IID, shard, Dirichlet partitions and client loaders;
- `src/aggregate.py` — weighted FedAvg aggregation;
- `src/training.py` — client updates, evaluation, and federated loop.

The notebook is responsible for:

1. choosing $\alpha$;
2. validating and saving each partition;
3. running controlled experiments;
4. saving histories;
5. analyzing the heterogeneity–performance relationship.


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_best_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_mnist(root):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_ds = datasets.MNIST(
        root=root,
        train=True,
        download=True,
        transform=transform,
    )

    test_ds = datasets.MNIST(
        root=root,
        train=False,
        download=True,
        transform=transform,
    )

    return train_ds, test_ds


def verify_partition(client_indices, dataset_size: int):
    """Verify exact coverage, no overlap, valid indices, and non-empty clients."""
    flat = [
        int(idx)
        for indices in client_indices.values()
        for idx in indices
    ]

    checks = {
        "dataset_size": dataset_size,
        "assigned": len(flat),
        "unique": len(set(flat)),
        "all_clients_nonempty": all(len(indices) > 0 for indices in client_indices.values()),
    }

    if checks["assigned"] != dataset_size:
        raise ValueError("Partition lost or added examples.")

    if checks["unique"] != dataset_size:
        raise ValueError("Partition contains duplicate or missing indices.")

    if not checks["all_clients_nonempty"]:
        raise ValueError("At least one client is empty.")

    if min(flat) < 0 or max(flat) >= dataset_size:
        raise ValueError("Partition contains invalid dataset indices.")

    return checks


def load_torch_object(path, map_location="cpu"):
    """Compatibility helper across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def alpha_tag(alpha: float) -> str:
    """Filesystem-friendly representation: 0.5 -> 0p5."""
    return f"{alpha:g}".replace(".", "p")


## 2. Experiment configuration

In [ ]:

@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    num_clients: int = 5

    # Heterogeneity-severity sweep
    alpha_values: tuple = (10.0, 1.0, 0.5, 0.1)
    anchor_alpha: float = 0.1

    # Hold E fixed initially so alpha is the experimental variable.
    local_epochs: int = 1
    num_rounds: int = 20

    batch_size: int = 64
    learning_rate: float = 0.01

    datasets_dir: str = "datasets"
    iid_results_dir: str = "results/iid_baseline"
    results_dir: str = "results/dirichlet_noniid"


CFG = ExperimentConfig()

RESULT_DIR = REPO_ROOT / CFG.results_dir
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = RESULT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

with (RESULT_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(asdict(CFG), f, indent=2)

device = get_best_device()

print(CFG)
print("Device :", device)
print("Results:", RESULT_DIR)


## 3. Load MNIST

In [ ]:

set_seed(CFG.seed)

train_ds, test_ds = load_mnist(REPO_ROOT / CFG.datasets_dir)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
)

print("Training examples:", len(train_ds))
print("Test examples    :", len(test_ds))



## 4. Verify the common initial checkpoint

The client partition must **not** change the initial global model.

All conditions therefore start from the same saved checkpoint $w_0$ used in the IID
baseline.


In [ ]:

INITIAL_MODEL_PATH = REPO_ROOT / CFG.iid_results_dir / "initial_model.pt"

if not INITIAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing initial checkpoint:\n{INITIAL_MODEL_PATH}\n"
        "Experiment 03 must use the same w0 as Experiment 01."
    )

initial_state = load_torch_object(INITIAL_MODEL_PATH)

initial_model = SimpleMLP(784, 10)
initial_model.load_state_dict(initial_state)
initial_model.to(device)

loss_fn = nn.CrossEntropyLoss()

initial_loss, initial_acc = evaluate(
    initial_model,
    test_loader,
    loss_fn,
    device,
)

print(f"Initial loss    : {initial_loss:.6f}")
print(f"Initial accuracy: {initial_acc:.4f}")

# These tolerances catch the wrong checkpoint/preprocessing without being brittle.
assert abs(initial_loss - 2.313339) < 1e-3
assert abs(initial_acc - 0.1267) < 1e-4



Expected result:

$$
\text{loss}\approx2.313339,\qquad
\text{accuracy}=12.67\%.
$$

Matching this value confirms that Experiment 03 begins from the same $w_0$ as the
previous experiments.



## 5. Build, validate, and save a Dirichlet partition

For each digit class, `partition_dirichlet()` samples a new client-allocation vector from

$$
\operatorname{Dirichlet}(\alpha,\ldots,\alpha).
$$

A partition is considered valid only if:

$$
\text{assigned}
=
\text{unique}
=
60{,}000,
$$

and every client receives at least one example.

Each $\alpha$ gets its own result directory so that partitions and histories cannot be
accidentally mixed.


In [ ]:

def alpha_result_dir(alpha: float) -> Path:
    path = RESULT_DIR / f"alpha_{alpha_tag(alpha)}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def prepare_partition(alpha: float, force_recreate: bool = False):
    """
    Load an exact saved partition if available; otherwise create, validate,
    inspect, and save a new Dirichlet partition.
    """
    alpha_dir = alpha_result_dir(alpha)
    partition_path = alpha_dir / "client_indices.pt"
    distribution_path = alpha_dir / "client_label_distribution.csv"

    if partition_path.exists() and not force_recreate:
        client_indices = load_torch_object(partition_path)
        source = "loaded"
    else:
        client_indices = partition_dirichlet(
            train_ds,
            num_clients=CFG.num_clients,
            alpha=alpha,
            seed=CFG.seed,
        )
        torch.save(client_indices, partition_path)
        source = "created"

    checks = verify_partition(client_indices, len(train_ds))

    distribution_df = client_label_counts(
        train_ds,
        client_indices,
    )
    distribution_df.to_csv(distribution_path, index=False)

    print(
        f"alpha={alpha:g}: {source} partition | "
        f"assigned={checks['assigned']} | unique={checks['unique']}"
    )

    return client_indices, distribution_df, checks



## 6. Inspect the strong-skew anchor condition: $\alpha=0.1$

Before launching a sweep, inspect one clearly heterogeneous partition carefully.

This is a **data validation step**, not a training result.


In [ ]:

anchor_indices, anchor_distribution_df, anchor_checks = prepare_partition(
    CFG.anchor_alpha
)

print("\nClient sizes:")
for client_id, indices in anchor_indices.items():
    print(f"Client {client_id}: {len(indices)} samples")

anchor_distribution_df


In [ ]:

digit_columns = [f"digit_{d}" for d in range(10)]

anchor_proportions = (
    anchor_distribution_df[digit_columns]
    .div(anchor_distribution_df["samples"], axis=0)
)

top2_share = np.sort(
    anchor_proportions.to_numpy(),
    axis=1,
)[:, -2:].sum(axis=1)

anchor_concentration_df = pd.DataFrame({
    "client": anchor_distribution_df["client"],
    "samples": anchor_distribution_df["samples"],
    "top_2_label_share": top2_share,
})

min_size = int(anchor_distribution_df["samples"].min())
max_size = int(anchor_distribution_df["samples"].max())

print(f"Smallest client: {min_size}")
print(f"Largest client : {max_size}")
print(f"Size ratio     : {max_size / min_size:.2f}x")

anchor_concentration_df


In [ ]:

fig, ax = plt.subplots(figsize=(9, 4))

image = ax.imshow(
    anchor_proportions.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
)

ax.set_xticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticks(range(CFG.num_clients))
ax.set_yticklabels([f"Client {i}" for i in range(CFG.num_clients)])

ax.set_xlabel("Digit label")
ax.set_ylabel("Client")
ax.set_title(f"Client label proportions — Dirichlet alpha={CFG.anchor_alpha:g}")

fig.colorbar(image, ax=ax, label="Proportion of client data")

plt.tight_layout()

fig.savefig(
    FIGURE_DIR / f"label_proportions_alpha_{alpha_tag(CFG.anchor_alpha)}.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()



### Development result already validated for $\alpha=0.1$, seed 42

The current implementation produced client sizes:

| Client | Samples | Top-2-label share |
|---:|---:|---:|
| 0 | 8,505 | 90.6% |
| 1 | 8,904 | 67.1% |
| 2 | 13,325 | 65.9% |
| 3 | 21,164 | 57.0% |
| 4 | 8,102 | 87.5% |

The partition covered all 60,000 MNIST training examples exactly once.

This confirms two properties:

1. **Strong label concentration:** several clients obtain most of their local data from only
   a few classes.
2. **Incidental quantity imbalance:** client sizes range from 8,102 to 21,164 examples
   (about a 2.61× ratio).

Therefore the correct interpretation is:

> This standard Dirichlet construction primarily controls label heterogeneity, while also
> inducing some client-size variation.

It would be incorrect to claim that this condition isolates pure label skew with equal
client quantities.


## 7. Create loaders for the anchor condition

In [ ]:

anchor_loaders = create_client_loaders(
    train_ds,
    anchor_indices,
    batch_size=CFG.batch_size,
)

for client_id, loader in anchor_loaders.items():
    print(
        f"Client {client_id}: "
        f"{len(loader.dataset)} samples, "
        f"{len(loader)} batches"
    )


## 8. Experiment runner

In [ ]:

def run_experiment(
    client_loaders,
    local_epochs: int,
    num_rounds: int,
    verbose: bool = True,
):
    """Run FedAvg from the exact common initial state."""
    set_seed(CFG.seed)

    model = SimpleMLP(784, 10)
    model.load_state_dict(copy.deepcopy(initial_state))
    model.to(device)

    start = time.perf_counter()

    trained_model, history = federated_train(
        global_model=model,
        client_loaders=client_loaders,
        num_rounds=num_rounds,
        local_epochs=local_epochs,
        learning_rate=CFG.learning_rate,
        loss_fn=nn.CrossEntropyLoss(),
        test_loader=test_loader,
        device=device,
        verbose=verbose,
    )

    runtime = time.perf_counter() - start

    return trained_model, history, runtime


def validate_history(history: pd.DataFrame, num_rounds: int):
    required_columns = {"round", "test_loss", "test_accuracy"}

    if not required_columns.issubset(history.columns):
        raise ValueError(
            f"History is missing columns: "
            f"{required_columns.difference(history.columns)}"
        )

    expected_rounds = list(range(num_rounds + 1))
    actual_rounds = history["round"].astype(int).tolist()

    if actual_rounds != expected_rounds:
        raise ValueError(
            f"Expected rounds 0..{num_rounds}, got {actual_rounds}"
        )

    if abs(float(history.iloc[0]["test_loss"]) - initial_loss) > 1e-3:
        raise ValueError("History does not begin from the common initial loss.")

    if abs(float(history.iloc[0]["test_accuracy"]) - initial_acc) > 1e-4:
        raise ValueError("History does not begin from the common initial accuracy.")

    return history



## 9. Smoke test

The smoke test checks only that:

$$
\text{Dirichlet partition}
\rightarrow
\text{client loaders}
\rightarrow
\text{local SGD}
\rightarrow
\text{FedAvg}
\rightarrow
\text{global evaluation}
$$

runs end-to-end.

It is deliberately short and is **not** a scientific result.


In [ ]:

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    _, smoke_history, smoke_runtime = run_experiment(
        client_loaders=anchor_loaders,
        local_epochs=1,
        num_rounds=2,
        verbose=True,
    )

    display(smoke_history)
    print(f"Runtime: {smoke_runtime:.1f}s")
else:
    print("Smoke test skipped. A development smoke test has already passed.")



### Previously validated development smoke test

For $\alpha=0.1,\ E=1,\ T=2$:

| Round | Test loss | Test accuracy |
|---:|---:|---:|
| 0 | 2.313339 | 12.67% |
| 1 | 1.826554 | 25.37% |
| 2 | 1.384799 | 50.63% |

The recorded runtime was about 64 seconds on that development environment. Runtime is
hardware-dependent and is not used as a scientific conclusion here.



## 10. Main experiment — heterogeneity-severity sweep

The main experiment varies only

$$
\alpha \in \{10,\;1,\;0.5,\;0.1\},
$$

while keeping

$$
E=1,\qquad T=20.
$$

For **every** $\alpha$:

1. build or load that alpha's exact partition;
2. validate coverage and client sizes;
3. rebuild client loaders;
4. reset the model to the same $w_0$;
5. run FedAvg;
6. save the round-by-round history immediately.

This is important: each $\alpha$ defines a different client dataset, so loaders must not
be reused across alpha values.


In [ ]:

def history_path(alpha: float) -> Path:
    return (
        alpha_result_dir(alpha)
        / f"fedavg_E{CFG.local_epochs}_history.csv"
    )


def run_alpha_sweep(
    alpha_values,
    overwrite: bool = False,
    verbose: bool = True,
):
    histories = {}
    runtimes = {}
    partition_stats = []

    for alpha in alpha_values:
        print("\n" + "=" * 64)
        print(f"alpha={alpha:g}")

        client_indices, distribution_df, checks = prepare_partition(alpha)

        client_loaders = create_client_loaders(
            train_ds,
            client_indices,
            batch_size=CFG.batch_size,
        )

        sizes = distribution_df["samples"].to_numpy()

        partition_stats.append({
            "alpha": alpha,
            "min_client_size": int(sizes.min()),
            "max_client_size": int(sizes.max()),
            "mean_client_size": float(sizes.mean()),
            "std_client_size": float(sizes.std(ddof=0)),
            "max_min_size_ratio": float(sizes.max() / sizes.min()),
        })

        path = history_path(alpha)

        if path.exists() and not overwrite:
            history = pd.read_csv(path)
            history = validate_history(history, CFG.num_rounds)

            histories[alpha] = history
            runtimes[alpha] = np.nan

            print(f"Loaded existing history: {path.name}")
            continue

        _, history, runtime = run_experiment(
            client_loaders=client_loaders,
            local_epochs=CFG.local_epochs,
            num_rounds=CFG.num_rounds,
            verbose=verbose,
        )

        history = validate_history(history, CFG.num_rounds)
        history.to_csv(path, index=False)

        histories[alpha] = history
        runtimes[alpha] = runtime

        print(f"Saved: {path}")
        print(f"Runtime: {runtime / 60:.2f} min")

    partition_stats_df = pd.DataFrame(partition_stats)
    partition_stats_df.to_csv(
        RESULT_DIR / "partition_size_summary.csv",
        index=False,
    )

    return histories, runtimes, partition_stats_df


In [ ]:

RUN_FULL_SWEEP = False
OVERWRITE_EXISTING = False

if RUN_FULL_SWEEP:
    histories, runtimes, partition_stats_df = run_alpha_sweep(
        CFG.alpha_values,
        overwrite=OVERWRITE_EXISTING,
        verbose=True,
    )
else:
    print(
        "Full alpha sweep disabled by default.\n"
        "Set RUN_FULL_SWEEP=True when you are ready to use compute."
    )



## 11. Load whatever completed alpha histories are available

This lets the analysis cells work after a partial run or after restarting the notebook.


In [ ]:

def load_available_histories():
    histories = {}

    for alpha in CFG.alpha_values:
        path = history_path(alpha)

        if path.exists():
            history = pd.read_csv(path)
            histories[alpha] = validate_history(
                history,
                CFG.num_rounds,
            )

    return histories


histories = load_available_histories()

print(
    "Available histories:",
    [f"alpha={alpha:g}" for alpha in histories]
)


## 12. Summarize completed runs

In [ ]:

def first_round_at_accuracy(history, threshold):
    hit = history.loc[
        history["test_accuracy"] >= threshold,
        "round",
    ]

    return int(hit.iloc[0]) if len(hit) else np.nan


summary_rows = []

for alpha, history in histories.items():
    summary_rows.append({
        "alpha": alpha,
        "local_epochs": CFG.local_epochs,
        "num_rounds": CFG.num_rounds,
        "final_loss": float(history.iloc[-1]["test_loss"]),
        "final_accuracy": float(history.iloc[-1]["test_accuracy"]),
        "round_to_70": first_round_at_accuracy(history, 0.70),
        "round_to_80": first_round_at_accuracy(history, 0.80),
        "round_to_90": first_round_at_accuracy(history, 0.90),
    })


summary_df = pd.DataFrame(summary_rows)

if len(summary_df):
    summary_df = summary_df.sort_values("alpha", ascending=False)
    summary_df.to_csv(
        RESULT_DIR / "summary.csv",
        index=False,
    )
    display(summary_df)
else:
    print("No full alpha histories are available yet.")


## 13. Accuracy vs communication round

In [ ]:

if histories:
    fig, ax = plt.subplots(figsize=(8, 5))

    for alpha in sorted(histories, reverse=True):
        history = histories[alpha]

        ax.plot(
            history["round"],
            history["test_accuracy"],
            marker="o",
            markersize=3,
            label=f"alpha={alpha:g}",
        )

    ax.set_xlabel("Communication round")
    ax.set_ylabel("Test accuracy")
    ax.set_title(
        f"FedAvg under Dirichlet label skew "
        f"(E={CFG.local_epochs})"
    )
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / "accuracy_vs_round_alpha_sweep.png",
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()
else:
    print("Run or restore at least one full alpha history first.")


## 14. Final accuracy vs Dirichlet concentration

In [ ]:

if len(summary_df):
    plot_df = summary_df.sort_values("alpha")

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.plot(
        plot_df["alpha"],
        plot_df["final_accuracy"],
        marker="o",
    )

    ax.set_xscale("log")
    ax.set_xlabel("Dirichlet concentration alpha (log scale)")
    ax.set_ylabel("Final test accuracy")
    ax.set_title(
        f"Final FedAvg accuracy vs heterogeneity severity "
        f"(E={CFG.local_epochs}, T={CFG.num_rounds})"
    )
    ax.grid(alpha=0.3)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / "final_accuracy_vs_alpha.png",
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()
else:
    print("No completed alpha sweep results yet.")



### How to interpret this plot

Remember:

$$
\alpha \downarrow
\quad\Longrightarrow\quad
\text{stronger heterogeneity}.
$$

Therefore, if FedAvg is sensitive to statistical heterogeneity, we expect performance to
degrade as $\alpha$ becomes smaller.

Do not infer a smooth monotonic law from a single seed automatically. The first sweep is
for establishing the phenomenon. If the pattern is meaningful, the final comparison should
later be repeated across multiple seeds and reported with uncertainty.


## 15. Quantify the quantity-imbalance confound

In [ ]:

partition_stat_rows = []

for alpha in CFG.alpha_values:
    alpha_dir = alpha_result_dir(alpha)
    distribution_path = alpha_dir / "client_label_distribution.csv"

    if not distribution_path.exists():
        continue

    df = pd.read_csv(distribution_path)
    sizes = df["samples"].to_numpy()

    partition_stat_rows.append({
        "alpha": alpha,
        "min_client_size": int(sizes.min()),
        "max_client_size": int(sizes.max()),
        "mean_client_size": float(sizes.mean()),
        "std_client_size": float(sizes.std(ddof=0)),
        "max_min_size_ratio": float(sizes.max() / sizes.min()),
    })


partition_stats_df = pd.DataFrame(partition_stat_rows)

if len(partition_stats_df):
    partition_stats_df = partition_stats_df.sort_values(
        "alpha",
        ascending=False,
    )

    partition_stats_df.to_csv(
        RESULT_DIR / "partition_size_summary.csv",
        index=False,
    )

    display(partition_stats_df)
else:
    print("No saved Dirichlet partitions found yet.")


In [ ]:

if len(partition_stats_df):
    plot_df = partition_stats_df.sort_values("alpha")

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.plot(
        plot_df["alpha"],
        plot_df["max_min_size_ratio"],
        marker="o",
    )

    ax.set_xscale("log")
    ax.set_xlabel("Dirichlet concentration alpha (log scale)")
    ax.set_ylabel("Largest / smallest client size")
    ax.set_title("Client-size imbalance induced by Dirichlet allocation")
    ax.grid(alpha=0.3)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / "client_size_imbalance_vs_alpha.png",
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()
else:
    print("No partition statistics available yet.")



## 16. Research interpretation

The main claim this notebook is designed to support is deliberately narrow:

> **Holding the model, optimizer, initialization, participation, local epochs, and number of
> communication rounds fixed, how does FedAvg performance change as Dirichlet concentration
> $\alpha$ changes?**

What this experiment **can** show:

- whether stronger statistical heterogeneity is associated with slower or worse global
  convergence;
- whether the performance gap grows as $\alpha$ decreases;
- how much incidental client-size imbalance accompanies each standard Dirichlet partition.

What this experiment **does not yet prove**:

- that optimization *client drift* is the sole mechanism behind any degradation;
- that the observed relationship generalizes across random seeds;
- that label skew and quantity skew have been perfectly separated.

A stronger next diagnostic is to measure client-update direction agreement. If

$$
\Delta_k = w_k - w_t,
$$

then pairwise cosine similarity

$$
\cos(\Delta_i,\Delta_j)
$$

can test whether updates become less aligned as heterogeneity increases.



## 17. Next steps

After the first $\alpha$-sweep is complete:

1. inspect accuracy/loss curves before adding more complexity;
2. add client-update cosine similarity as a heterogeneity diagnostic;
3. repeat the key comparison across several seeds only after the phenomenon is established;
4. implement a separate quantity-skew experiment;
5. then introduce FedProx under the **same saved partitions** to test whether regularization
   mitigates heterogeneity.

Do not mix FedProx into this notebook. Experiment 03 should remain the clean FedAvg
heterogeneity-severity baseline.


## 18. Artifact check

In [ ]:

artifact_rows = [
    {
        "artifact": str((RESULT_DIR / "config.json").relative_to(REPO_ROOT)),
        "exists": (RESULT_DIR / "config.json").exists(),
    },
    {
        "artifact": str(
            (
                alpha_result_dir(CFG.anchor_alpha)
                / "client_indices.pt"
            ).relative_to(REPO_ROOT)
        ),
        "exists": (
            alpha_result_dir(CFG.anchor_alpha)
            / "client_indices.pt"
        ).exists(),
    },
    {
        "artifact": str(
            (
                alpha_result_dir(CFG.anchor_alpha)
                / "client_label_distribution.csv"
            ).relative_to(REPO_ROOT)
        ),
        "exists": (
            alpha_result_dir(CFG.anchor_alpha)
            / "client_label_distribution.csv"
        ).exists(),
    },
    {
        "artifact": str((RESULT_DIR / "summary.csv").relative_to(REPO_ROOT)),
        "exists": (RESULT_DIR / "summary.csv").exists(),
    },
]

artifact_check_df = pd.DataFrame(artifact_rows)
artifact_check_df
